# Gradio Day!

Today we will build User Interfaces using the outrageously simple Gradio framework.

Prepare for joy!

Please note: your Gradio screens may appear in 'dark mode' or 'light mode' depending on your computer settings.

In [3]:
import os
from dotenv import load_dotenv
from openai import OpenAI

In [3]:
import gradio as gr 

In [4]:
# for key, value in os.environ.items():
#     print(f"{key}: {value}")
load_dotenv(override=True)
gpt_api_key = os.getenv("OPENAI_API_KEY")
gemini_api_key = os.getenv("GOOGLE_API_KEY") 

# for key, value in os.environ.items():
#     print(f"{key}: {value}")

In [5]:
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
gpt_model = "gpt-4.1-mini"
gemini_model = "gemini-flash-latest"

gpt = OpenAI()
gemini = OpenAI(base_url=gemini_url, api_key=gemini_api_key)

In [6]:
system_message = "You are a helpful assistant"

def message_gpt(prompt):
    messages = [{"role":"system","content":system_message},{"role":"user", "content":prompt}]
    response = gpt.chat.completions.create(model = gpt_model, messages= messages)
    return response.choices[0].message.content

In [7]:
def message_gemini(prompt):
    messages = [{"role":"system","content":system_message},{"role":"user", "content":prompt}]
    response = gemini.chat.completions.create(model=gemini_model, messages=messages)
    return response.choices[0].message.content

In [15]:
message_gpt("What is todays date")

"Today's date is April 27, 2024."

## User Interface time!

In [8]:
def shout(text):
    print(f"Shout has been called with  input {text}")
    return text.upper()

In [9]:
shout("Hello")

Shout has been called with  input Hello


'HELLO'

In [10]:
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">NOTE: Using Gradio's Share tool</h2>
            <span style="color:#900;">I'm about to show you a really cool way to share your Gradio UI with others. This deploys your gradio app as a demo on gradio's website, and then allows gradio to call the 'shout' function. This uses an advanced technology known as 'HTTP tunneling' (like ngrok for people who know it) which isn't allowed by many Antivirus programs and corporate environments. If you get an error, just skip the next cell.<br/>
            </span>
        </td>
    </tr>
</table>

In [11]:
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(share=True)

* Running on local URL:  http://127.0.0.1:7861

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


In [12]:
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


## Adding authentication

Gradio makes it very easy to have userids and passwords

Obviously if you use this, have it look properly in a secure place for passwords! At a minimum, use your .env

In [13]:
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(inbrowser=True, auth=("zw","abc"))

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


## Forcing dark mode

Gradio appears in light mode or dark mode depending on the settings of the browser and computer. There is a way to force gradio to appear in dark mode, but Gradio recommends against this as it should be a user preference (particularly for accessibility reasons). But if you wish to force dark mode for your screens, below is how to do it.

In [14]:
message_input = gr.Textbox(label="Your Message", info="Enter a message gor gemini", lines=7)
message_output = gr.Textbox(label="Response", lines=10)

view = gr.Interface(
    fn= message_gemini,
    title = "GEMINI",
    inputs=[message_input],
    outputs=[message_output],
    examples=["Hello", "Howdy"],
    flagging_mode="never"
    )
view.launch(inbrowser=True, auth=[("zw","123"),("zw2","1232")])

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


In [23]:
system_message = "You are a helpful assistant that responds in mardown without code blocks"

message_input = gr.Textbox(label="Your Message", info="Enter a message for gemini", lines=7)
message_output = gr.Markdown(label="Response")

view = gr.Interface(
    fn = message_gemini,
    title = "GEMINI",
    inputs=[message_input],
    outputs=[message_output],
    flagging_mode="never"
)
view.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


In [34]:
def stream_gemini(prompt):
    message = [
        {"role":"system","content":system_message},
        {"role":"user","content":prompt}
    ]
    stream = gemini.chat.completions.create(model=gemini_model,messages=message, stream=True)
    result=""
    for chunk in stream:
        content = chunk.choices[0].delta.content or ""
        result+=content
        yield result

In [35]:
system_message = "You are a helpful assistant that responds in mardown without code blocks"

message_input = gr.Textbox(label="Your Message", info="Enter a message for gemini", lines=7)
message_output = gr.Markdown(label="Response")

view = gr.Interface(
    fn = stream_gemini,
    title = "GEMINI",
    inputs=[message_input],
    outputs=[message_output],
    flagging_mode="never"
)
view.launch()

* Running on local URL:  http://127.0.0.1:7875
* To create a public link, set `share=True` in `launch()`.


In [37]:
def stream_gpt(prompt):
    message = [
        {"role":"system","content":system_message},
        {"role":"user","content":prompt}
    ]
    stream = gpt.chat.completions.create(model=gpt_model,messages=message, stream=True)
    result=""
    for chunk in stream:
        content = chunk.choices[0].delta.content or ""
        result+=content
        yield result

In [38]:
message_input = gr.Textbox(label="Your Message:", info="Enter a message for GPT", lines=7)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn = stream_gpt,
    title = "GPT",
    inputs=[message_input],
    outputs=[message_output],
    flagging_mode="never"
)
view.launch()

* Running on local URL:  http://127.0.0.1:7876
* To create a public link, set `share=True` in `launch()`.


## And now getting fancy

Remember to check the Intermediate Python Guide if you're unsure about generators and "yield"

In [40]:
def stream_model(prompt,model):
    if model == "GPT":
        result = stream_gpt(prompt)
    elif model == "GEMINI":
        result = stream_gemini(prompt)
    else:
        raise ValueError("UnknownModel")
    yield from result


In [44]:
message_input = gr.Textbox(label="Your Message:", info ="Enter a message for LLM", lines=7)
message_output = gr.Markdown(label="Response")
model_selector = gr.Dropdown(["GPT","GEMINI"],label="Select Model", value="GEMINI")

view = gr.Interface(
    fn = stream_model,
    title = "LLMs",
    inputs=[message_input,model_selector],
    outputs=[message_output],
    flagging_mode="never"
)
view.launch()

* Running on local URL:  http://127.0.0.1:7879
* To create a public link, set `share=True` in `launch()`.


# Building a company brochure generator

Now you know how - it's simple!

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you read the next few cells</h2>
            <span style="color:#900;">
                Try to do this yourself - go back to the company brochure in week1, day5 and add a Gradio UI to the end. Then come and look at the solution.
            </span>
        </td>
    </tr>
</table>

In [48]:
from scraper import fetch_website_contents

In [49]:
# Again this is typical Experimental mindset - I'm changing the global variable we used above:

system_message = """
You are an assistant that analyzes the contents of a company website landing page
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
"""

In [50]:
def stream_brochure(company_name, url, model):
    yield ""
    prompt = f"Please generate a company brochure for {company_name}. Here is their landing page:\n"
    prompt += fetch_website_contents(url)
    if model=="GPT":
        result = stream_gpt(prompt)
    elif model=="GEMINI":
        result = stream_gemini(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

In [53]:
name_input = gr.Textbox(label="Company name:")
url_input = gr.Textbox(label="Landing page URL including http:// or https://")
model_selector = gr.Dropdown(["GPT", "GEMINI"], label="Select model", value="GEMINI")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_brochure,
    title="Brochure Generator", 
    inputs=[name_input, url_input, model_selector], 
    outputs=[message_output], 
    examples=[
            ["Hugging Face", "https://huggingface.co", "GPT"],
            ["Edward Donner", "https://edwarddonner.com", "GEMINI"]
        ], 
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7883
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "d:\LLM_AI_Engineering\.venv\Lib\site-packages\urllib3\connection.py", line 198, in _new_conn
    sock = connection.create_connection(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\LLM_AI_Engineering\.venv\Lib\site-packages\urllib3\util\connection.py", line 60, in create_connection
    for res in socket.getaddrinfo(host, port, family, socket.SOCK_STREAM):
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Dell Latitude E7270\AppData\Roaming\uv\python\cpython-3.12.12-windows-x86_64-none\Lib\socket.py", line 978, in getaddrinfo
    for res in _socket.getaddrinfo(host, port, family, type, proto, flags):
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
socket.gaierror: [Errno 11002] getaddrinfo failed

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "d:\LLM_AI_Engineering\.venv\Lib\site-packages\urllib3\connectionp

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Gradio Resources</h2>
            <span style="color:#f71;">If you'd like to go deeper on Gradio, check out the amazing documentation - a wonderful rabbit hole.<br/>
            <a href="https://www.gradio.app/guides/quickstart">https://www.gradio.app/guides/quickstart</a><br/>Gradio is primarily designed for Demos, Prototypes and MVPs, but I've also used it frequently to make internal apps for power users.
            </span>
        </td>
    </tr>
</table>